In [1]:
import yfinance as yf
import atoti as tt
import pandas as pd
from datetime import datetime

tickers = [
    'GC=F', 'SI=F', 'CL=F', 'NG=F', 'HG=F', # Komoditas (Gold, Silver, Oil, Gas, Copper)
    'BTC-USD',                              # Aset Kripto
    'NVDA', 'AAPL', 'MSFT', 'TSLA', 'GOOGL',# Saham Teknologi (NVidia, Apple, Microsoft, Tesla, Alphabet)
    'BBCA.JK', 'BBRI.JK', 'TLKM.JK', 'ASII.JK', # Saham Indonesia (BBCA, BBRI, TLKM, ASII)
    '^GSPC', '^JKSE', '^IXIC', 'IDR=X', '^TNX'  # Indeks dan Mata Uang (S&P 500, IDX Composite, NASDAQ, IDR/USD, US Treasury Yield)
]

print("Mengumpulkan data dari Yahoo Finance")


data = yf.download(tickers, start="1927-01-01")['Close']

df = data.stack().reset_index()
df.columns = ['Date', 'Symbol', 'Price']
df['Date'] = pd.to_datetime(df['Date'])

print(f"Berhasil menarik {len(df)} baris data")
df

Welcome to Atoti 0.9.13!

By using this community edition, you agree with the license available at https://docs.activeviam.com/products/atoti/python-sdk/latest/eula.html.
Browse the official documentation at https://docs.activeviam.com/products/atoti/python-sdk.
Join the community at https://www.atoti.io/register.

Atoti collects telemetry data, which is used to help understand how to improve the product.
If you don't wish to send usage data, you can request a trial license at https://www.atoti.io/evaluation-license-request.

You can hide this message by setting the `ATOTI_HIDE_EULA_MESSAGE` environment variable to True.
Mengumpulkan data dari Yahoo Finance


[*********************100%***********************]  20 of 20 completed

Berhasil menarik 166060 baris data


,Date,Symbol,Price
0,1927-12-30,^GSPC,17.660000
1,1928-01-03,^GSPC,17.760000
2,1928-01-04,^GSPC,17.719999
3,1928-01-05,^GSPC,17.549999
4,1928-01-06,^GSPC,17.660000
...,...,...,...
166055,2026-03-16,IDR=X,16988.000000
166056,2026-03-16,NG=F,3.107000
166057,2026-03-16,SI=F,80.245003
166058,2026-03-16,TLKM.JK,2930.000000


In [2]:

# --- SEL 2: MEMBANGUN CUBE & METRICS ---

asset_mapping = {
    'GC=F': 'Komoditas', 'SI=F': 'Komoditas', 'CL=F': 'Komoditas', 'NG=F': 'Komoditas', 'HG=F': 'Komoditas',
    'BTC-USD': 'Kripto',
    'NVDA': 'Saham Teknologi', 'AAPL': 'Saham Teknologi', 'MSFT': 'Saham Teknologi', 'TSLA': 'Saham Teknologi', 'GOOGL': 'Saham Teknologi',
    'BBCA.JK': 'Saham Indonesia', 'BBRI.JK': 'Saham Indonesia', 'TLKM.JK': 'Saham Indonesia', 'ASII.JK': 'Saham Indonesia',
    '^GSPC': 'Indeks & Mata Uang', '^JKSE': 'Indeks & Mata Uang', '^IXIC': 'Indeks & Mata Uang', 'IDR=X': 'Indeks & Mata Uang', '^TNX': 'Indeks & Mata Uang'
}

df['Kategori'] = df['Symbol'].map(asset_mapping)

df

,Date,Symbol,Price,Kategori
0,1927-12-30,^GSPC,17.660000,Indeks & Mata Uang
1,1928-01-03,^GSPC,17.760000,Indeks & Mata Uang
2,1928-01-04,^GSPC,17.719999,Indeks & Mata Uang
3,1928-01-05,^GSPC,17.549999,Indeks & Mata Uang
4,1928-01-06,^GSPC,17.660000,Indeks & Mata Uang
...,...,...,...,...
166055,2026-03-16,IDR=X,16988.000000,Indeks & Mata Uang
166056,2026-03-16,NG=F,3.107000,Komoditas
166057,2026-03-16,SI=F,80.245003,Komoditas
166058,2026-03-16,TLKM.JK,2930.000000,Saham Indonesia


In [3]:

# 2. Mulai Session Atoti
session = tt.Session.start()
macro_table = session.read_pandas(df, table_name="MarketData", keys=["Date", "Symbol"])



In [4]:
macro_table.head()

,,Price,Kategori
Date,Symbol,,
1927-12-30,^GSPC,17.66,Indeks & Mata Uang
1928-01-11,^GSPC,17.35,Indeks & Mata Uang
1928-02-06,^GSPC,17.450001,Indeks & Mata Uang
1928-02-07,^GSPC,17.440001,Indeks & Mata Uang
1928-02-27,^GSPC,17.110001,Indeks & Mata Uang


In [5]:
cube = session.create_cube(macro_table)
h = cube.hierarchies
l = cube.levels
m = cube.measures


# Membuat Hierarchy Waktu yang cerdas (Tahun > Bulan > Hari)
cube.create_date_hierarchy(
    "Periode Analisis", 
    column=macro_table["Date"],
    levels={
        "Tahun": "yyyy",
        "Bulan": "MMMM",
        "Hari": "dd"
    }
)

# Opsional: Hapus hierarchy 'Date' bawaan yang flat agar tidak bingung
del h["Date"]


print('Hierarchies:', list(h.keys()))
print('Levels:', list(l.keys()))
print('Measures:', list(m.keys()))

Hierarchies: [('MarketData', 'Kategori'), ('MarketData', 'Symbol'), ('MarketData', 'Periode Analisis')]
Levels: [('MarketData', 'Kategori', 'Kategori'), ('MarketData', 'Symbol', 'Symbol'), ('MarketData', 'Periode Analisis', 'Tahun'), ('MarketData', 'Periode Analisis', 'Bulan'), ('MarketData', 'Periode Analisis', 'Hari')]
Measures: ['Price.MEAN', 'Price.SUM', 'contributors.COUNT', 'update.TIMESTAMP']


HARGA ASLI

In [6]:
m["Price.VALUE"] = tt.agg.single_value(macro_table["Price"])


cube.query(
            m['Price.VALUE'], 
            levels=[l['Symbol'], l['Hari']])

Price.VALUE
Symbol Tahun Bulan    Hari            
AAPL   1980  December 12           .10
                      15           .09
                      16           .09
                      17           .09
                      18           .09
...                                ...
^TNX   2026  March    09          4.14
                      10          4.14
                      11          4.21
                      12          4.27
                      13          4.28

[166060 rows x 1 columns]

In [ ]:
m["Price H-1"] = tt.shift(
                        m["Price.VALUE"], 
                        h["Periode Analisis"],
                        offset=-1,
                        mode="measure")

cube.query(
            m['Price.VALUE'], 
            m['Price H-1'], 
            levels=[l['Symbol'], l['Hari']])

Price.VALUE Price H-1
Symbol Tahun Bulan    Hari                      
AAPL   1980  December 12           .10          
                      15           .09       .10
                      16           .09       .09
                      17           .09       .09
                      18           .09       .09
...                                ...       ...
^TNX   2026  March    09          4.14      4.13
                      10          4.14      4.14
                      11          4.21      4.14
                      12          4.27      4.21
                      13          4.28      4.27

[166060 rows x 2 columns]

In [8]:
m["Daily Return"] = ((m["Price.VALUE"] - m["Price H-1"]) / m["Price H-1"])
m["Daily Return"].formatter = "DOUBLE[0.00%]"
cube.query(m['Daily Return'], levels=[l['Symbol'], l['Hari']])

Daily Return
Symbol Tahun Bulan    Hari             
AAPL   1980  December 15         -5.22%
                      16         -7.34%
                      17          2.48%
                      18          2.90%
                      19          6.10%
...                                 ...
^TNX   2026  March    09          0.07%
                      10          0.00%
                      11          1.74%
                      12          1.54%
                      13          0.28%

[166040 rows x 1 columns]

In [9]:
# PERBAIKAN: Gunakan parameter 'level' dan arahkan ke l["Hari"]
# l["Hari"] adalah level terbawah dari hierarchy "Periode Analisis" kamu
cumulative_scope = tt.CumulativeScope(level=l["Hari"])

# Sekarang hitung ulang Cumulative Return
m["Cumulative Return"] = tt.agg.prod(
    1 + m["Daily Return"], 
    scope=cumulative_scope
) - 1

m["Cumulative Return"].formatter = "DOUBLE[0.00%]"

cube.query(
            m["Cumulative Return"], 
            levels=[l["Symbol"], l["Hari"]],
            )

Cumulative Return
Symbol Tahun Bulan    Hari                  
AAPL   1980  December 12               0.00%
                      15              -5.22%
                      16             -12.17%
                      17             -10.00%
                      18              -7.39%
...                                      ...
^TNX   2026  March    09               7.15%
                      10               7.15%
                      11               9.02%
                      12              10.70%
                      13              11.01%

[166060 rows x 1 columns]

In [12]:
# PASTIKAN SEL-SEL SEBELUMNYA (SESSION & CUBE CREATION) SUDAH DI-RUN ULANG!

# Perbaikan format window: gunakan "-P5D" untuk 5 hari
m['5-day SMA'] = tt.agg.mean(
    m["Price.VALUE"], 
    scope=tt.CumulativeScope(
        level=l["Hari"], 
        window=("-P5D", None)
    )
)

m['5-day SMA'].formatter = "DOUBLE[0.00]"

# Cek apakah sudah muncul
cube.query(m['5-day SMA'], levels=[l['Symbol'], l['Hari']])

GraphQLClientGraphQLMultiError: Cannot invoke "com.activeviam.activepivot.core.intf.internal.structure.IInternalActivePivotVersion.getHierarchies()" because "this.activePivot" is null (Error id: 00b1f9fb-e2d4-4d9a-a4ae-c57763c8db1f)
java.lang.NullPointerException: Cannot invoke "com.activeviam.activepivot.core.intf.internal.structure.IInternalActivePivotVersion.getHierarchies()" because "this.activePivot" is null
	at com.activeviam.activepivot.mdx.impl.internal.meta.impl.MdxCube.<init>(MdxCube.java:402)
	at com.activeviam.activepivot.mdx.impl.internal.meta.impl.MdxCube.<init>(MdxCube.java:366)
	at com.activeviam.activepivot.mdx.impl.internal.meta.impl.MdxCube.<init>(MdxCube.java:354)
	at com.activeviam.atoti.server.base.private_.pivot.graphql.service.DataModelService.getCubeMeasures(DataModelService.java:303)
	at com.activeviam.atoti.server.base.private_.pivot.graphql.service.DataModelService.getRuntimeMeasures(DataModelService.java:325)
	at com.activeviam.atoti.server.base.private_.pi

In [ ]:
# Menggunakan Window, bukan Scope
m["Volatility_Rolling_20D"] = tt.agg.std(
    m["Daily Return"],
    window=tt.Window(over=h["Periode Analisis"], offset="-20D", window_type="rolling")
)
m["Volatility_Rolling_20D"].formatter = "DOUBLE[0.00%]"

In [ ]:
# Ini akan mencetak link dashboard kamu
print(f"Buka dashboard di sini: {session.url}")

# Atau biarkan Atoti membukanya secara otomatis di browser default
session.link

In [ ]:

cube.query(m['Price.MEAN'], levels=[l['Symbol'], l['Bulan']])

In [ ]:

cube.query(m['Price.MEAN'], levels=[l['Symbol'], l['Tahun']])

1. Bagaimana performa kumulatif (Cumulative Return) dari kategori 'Saham Teknologi' dibandingkan 'Saham Indonesia' selama tiga krisis besar: The Great Depression (1929), Dot-com Bubble (2000), dan Pandemi COVID-19 (2020)?

In [11]:
cube.query(m['Price.MEAN'], levels=[l['Kategori'], l['Tahun']])

GraphQLClientGraphQLMultiError: Cannot invoke "com.activeviam.activepivot.core.intf.internal.structure.IInternalActivePivotVersion.getHierarchies()" because "this.activePivot" is null (Error id: 8c213e27-f06f-430f-9808-449559d96d87)
java.lang.NullPointerException: Cannot invoke "com.activeviam.activepivot.core.intf.internal.structure.IInternalActivePivotVersion.getHierarchies()" because "this.activePivot" is null
	at com.activeviam.activepivot.mdx.impl.internal.meta.impl.MdxCube.<init>(MdxCube.java:402)
	at com.activeviam.activepivot.mdx.impl.internal.meta.impl.MdxCube.<init>(MdxCube.java:366)
	at com.activeviam.activepivot.mdx.impl.internal.meta.impl.MdxCube.<init>(MdxCube.java:354)
	at com.activeviam.atoti.server.base.private_.pivot.graphql.service.DataModelService.getCubeMeasures(DataModelService.java:303)
	at com.activeviam.atoti.server.base.private_.pivot.graphql.service.DataModelService.getRuntimeMeasures(DataModelService.java:325)
	at com.activeviam.atoti.server.base.private_.pi

1. Analisis Ketahanan Sektoral terhadap Krisis Global (Slicing & Dicing)Pertanyaan: "Bagaimana performa kumulatif (Cumulative Return) dari kategori 'Saham Teknologi' dibandingkan 'Saham Indonesia' selama tiga krisis besar: The Great Depression (1929), Dot-com Bubble (2000), dan Pandemi COVID-19 (2020)?"Kenapa Menarik: Ini menunjukkan kekuatan Hierarchy di Atoti. Kamu bisa melakukan Slicing pada level Tahun dan Dicing pada level Kategori.Insight: Kamu bisa membuktikan apakah benar instrumen 'Komoditas' (Emas) selalu menjadi penyelamat saat sektor 'Saham' sedang hancur.2. Efisiensi Investasi: "The Sharpe Ratio Challenge" (Custom Measures)Pertanyaan: "Manakah dari 20 instrumen tersebut yang memiliki rasio keuntungan terhadap risiko (Volatility) terbaik dalam 10 tahun terakhir?"Kenapa Menarik: Dalam Data Warehouse, ini menguji kemampuanmu membuat Complex Measures. Kamu akan menghitung $Sharpe Ratio = \frac{\text{Mean Return}}{\text{Volatility}}$.Insight: Kamu bisa menggunakan Scatter Plot di Atoti untuk memetakan instrumen di area "High Return - Low Risk".3. Fenomena "Digital Gold" vs "Physical Gold" (Time-Series Comparison)Pertanyaan: "Sejak kemunculan Bitcoin (BTC-USD), apakah pergerakannya berkorelasi positif dengan Emas (GC=F) atau justru lebih mirip dengan pergerakan Saham Teknologi (NVDA/AAPL)?"Kenapa Menarik: Ini menguji Multi-Scenario Analysis. Kamu membandingkan dua instrumen dari kategori berbeda dalam satu window waktu yang sama.Insight: Memberikan jawaban berbasis data apakah Bitcoin benar-benar layak disebut "Emas Digital" atau sekadar aset spekulatif teknologi.4. Transmisi Kebijakan Moneter AS ke Pasar Lokal (Macro Analysis)Pertanyaan: "Bagaimana dampak kenaikan US Treasury Yield (^TNX) terhadap performa saham perbankan Indonesia (BBCA.JK & BBRI.JK)?"Kenapa Menarik: Ini menunjukkan hubungan antar Dimension yang berbeda (Macro-Index vs Indo-Stocks).Insight: Kamu bisa menganalisis apakah ada lagging effect (jeda waktu). Misalnya, saat bunga AS naik hari ini, apakah saham Indo baru bereaksi 2-3 hari kemudian? (Gunakan fungsi tt.shift).5. Deteksi Momentum: "Golden Cross Across Categories" (Rolling Windows)Pertanyaan: "Di kategori manakah fenomena Golden Cross (SMA 5 > SMA 20) paling sering menghasilkan keuntungan berkelanjutan dalam satu bulan berikutnya?"Kenapa Menarik: Ini menggunakan fungsi Moving Average yang baru saja kita buat. Ini menunjukkan bahwa Data Warehouse bisa digunakan untuk Predictive Analytics sederhana.Insight: Kamu bisa menyimpulkan apakah strategi teknikal ini lebih efektif di pasar 'Kripto' yang liar atau pasar 'Indeks' yang lebih stabil.